In [32]:
# Large module imports
import librosa
import numpy as np
import pandas as pd
from pathlib import Path



# Target Features

The paper cites that 20-30 features were used for calculating the target cost. The paper uses basic phoneme traits. The paper also looks at prosodic characteristics of the preceding and following phonemes.

Calculation of many of the feature vectors is quite involved, so subsets of feature vectors should be established. Additional features can be added and calculated as the experiment progresses.

Simple features describing phonemes: vowels or consonants, voicing, articulation manner, articulation place, pre/post silence markers. We'll also consider the proceeding and following phonemes and their duration.

## Feature Vector Shape

Features that comprise the vector and their shape.
- Vowel or consonant (binary option).
- Voicing (phoneme is voiced or not) (binary option).
- Articulation manner (plosive, fricative, affricate, nasal, liquid, glide, vowel, silence) (one-hot).
- Articulation place (bilabial, labiodental, dental, alveolar, postalveolar, palatal, velar, glottal, none) (one-hot).
- Proceeding phoneme (one-hot).
- Current phoneme (one-hot).
- Following phoneme (one-hot).
- Duration (float).

The first 4 features are known beforehand from a lookup table. The last 4 features can be read from each `lab/` file.

In [33]:
# Articulatory/Phonological mapping table for ARPAbet phonemes
# Categories:
# - is_vowel: 1 if vowel, 0 otherwise [cite: 39]
# - is_voiced: 1 if vocal cords vibrate, 0 otherwise [cite: 40]
# - manner: plosive, fricative, affricate, nasal, liquid, glide, vowel, silence [cite: 41]
# - place: bilabial, labiodental, dental, alveolar, postalveolar, palatal, velar, glottal, none [cite: 42]

# Vocabulary lists for categorical fields
MANNERS = ['vowel', 'plosive', 'fricative', 'affricate', 'nasal', 'liquid', 'glide', 'silence']
PLACES = ['bilabial', 'labiodental', 'dental', 'alveolar', 'postalveolar', 'palatal', 'velar', 'glottal', 'none']

PHONEME_FEATURES = {
    # Silence / Pauses
    'pau': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'silence', 'place': 'none'},
    'ssil': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'silence', 'place': 'none'},
    
    # Vowels (All vowels are voiced)
    'aa': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ae': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ah': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ao': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'aw': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ax': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ay': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'eh': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'er': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ey': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ih': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'iy': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'ow': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'oy': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'uh': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},
    'uw': {'is_vowel': 1, 'is_voiced': 1, 'manner': 'vowel', 'place': 'none'},

    # Stops / Plosives
    'p':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'bilabial'},
    'b':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'bilabial'},
    't':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'alveolar'},
    'd':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'alveolar'},
    'k':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'plosive', 'place': 'velar'},
    'g':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'plosive', 'place': 'velar'},

    # Fricatives
    'f':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'labiodental'},
    'v':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'labiodental'},
    'th': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'dental'},
    'dh': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'dental'},
    's':  {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'alveolar'},
    'z':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'alveolar'},
    'sh': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'postalveolar'},
    'zh': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'fricative', 'place': 'postalveolar'},
    'hh': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'fricative', 'place': 'glottal'},

    # Affricates
    'ch': {'is_vowel': 0, 'is_voiced': 0, 'manner': 'affricate', 'place': 'postalveolar'},
    'jh': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'affricate', 'place': 'postalveolar'},

    # Nasals
    'm':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'bilabial'},
    'n':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'alveolar'},
    'ng': {'is_vowel': 0, 'is_voiced': 1, 'manner': 'nasal', 'place': 'velar'},

    # Liquids & Glides
    'l':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'liquid', 'place': 'alveolar'},
    'r':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'liquid', 'place': 'alveolar'},
    'w':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'glide', 'place': 'bilabial'},
    'y':  {'is_vowel': 0, 'is_voiced': 1, 'manner': 'glide', 'place': 'palatal'},
}

In [34]:
# Calculating phoneme features from LUT and lab/ files

## Phoneme features can be grabbed directly from the LUT
## Additional features can be grabbed while reading the file for acoustic processing

# Acoustic Features

Acoustic features are required for training the weights of the target unit selection. Acoustic features are used for calculating concatenation costs. Weights for the acoustic features are known, but the features must still be calculated. Acoustic features used by the paper are cepstral distance at point of concatenation, and absolute differences in log power and pitch. 

## Acoustic Features Database

I need to decide the best way to store these features. There will be many instances of a given phoneme, with repetitions even within the same file.

## Constructing the Database

Three acoustic features to be calculated for each phoneme: log power, log pitch, and cepstral distance at the point of concatenation. The location and duration of each phoneme in the source audio file can be identified from the durations extracted from the $.lab$ files.

### Log Power and Log Pitch Calculation

Log Power Computation
- Extract phoneme location and duration from the `lab` file.
- Calculate RMS power as ln(RMS + epsilon) from the corresponding wave file.

Log Pitch Computation
- Extract phoneme location and duration from the `lab` file.
- Calculate $F0$ from the corresponding `pm_lab` file. Count periods within the phoneme's duration. $F0 = N/Dur$

### Cepstral Distance Calculation

Cepstrum Computation
- Extract phoneme location and duration from the `lab` file.
- Calculate the cepstrum from the corresponding wave file with $N=24$ points.

In [35]:
# Acoustic feature computations

def log_power(samples):
    e = 0.01
    return np.log(np.sqrt(np.mean(samples**2)) + e)

def log_pitch(periods, duration):
    return np.log(periods / duration + .0001)

def count_periods(pitchmarks, t_start, t_dur):
    def extract_line(line):
        t_curr, _ = line.strip().split(' ')
        return (float(t_curr))

    pitchmarks = pitchmarks[8:-1]
    count = 0
    for line in pitchmarks:
        t_curr = extract_line(line)
        if t_start <= t_curr <= t_start + t_dur:
            count += 1
        elif t_curr > t_start + t_dur:
            break
    return count

def cepstrum(samples, k=12):
    n_fft = 512
    n_pad = n_fft - len(samples)

    if n_pad > 0:
        samples = np.pad(samples, (0, n_pad))
    else:
        samples = samples[:n_fft]  # Take first 512 samples if longer

    # Compute MFCCs without centering/padding extra frames
    mfcc = librosa.feature.mfcc(
        y=samples,
        sr=16000,
        n_mfcc=k,
        n_fft=n_fft,
        hop_length=n_fft
        + 1,  # Prevents librosa from trying to frame into step 2
        center=False,  # Prevents zero-padding at boundaries
    )
    return mfcc.squeeze()


In [36]:
# Data processing aids

# Gather related files for acoustic processing (lab, wav, pm)
def gather_source_files(src_root):
    src_root = Path(src_root)
    lab_dir = src_root / "lab"

    file_triplets = []
    for lab_file in lab_dir.glob("*.lab"):
        stem = lab_file.stem

        triplet = {
            "ph": lab_file,
            "wav": src_root / "wav" / f"{stem}.wav",
            "pm": src_root / "pm" / f"{stem}.pm",
        }
        file_triplets.append(triplet)
    return file_triplets

# From a label file, returns an ordered list of phonemes with their:
# Current phoneme, next phoneme, previous phoneme, Start times, duration, and a unique id
# The id has format {file_stem}_{index:04d} so you can parse which file and
# which occurrence it came from.
def process_labeled_phonemes(lab_path):
    def proc_line(line):
        t_end, _, ph = line.strip().split(' ')
        return float(t_end), ph

    file_stem = Path(lab_path).stem

    phonemes = []
    with open(lab_path, "r", encoding="utf-8") as lab_file:
        lines = lab_file.readlines()
        for i in range(2, len(lines) - 2):
            t_end_cph, cph = proc_line(lines[i])
            t_end_pph, pph = proc_line(lines[i - 1])
            _, nph = proc_line(lines[i + 1])
            idx = len(phonemes)
            phonemes.append({'id'   : f'{file_stem}_{idx:04d}',
                             'ph'   : cph, 
                             'nph'  : nph,
                             'pph'  : pph,
                             'start': t_end_pph,
                             'dur'  : t_end_cph - t_end_pph})
    return phonemes


In [37]:
# Data processing loop
# We find the files we need, then construct features for each phoneme from data in the current file(s).

import pyarrow as pa
import pyarrow.parquet as pq

# Helper: write or append a DataFrame to a Parquet file
# pyarrow does not support in-place append, so we read + concat + rewrite
def write_parquet(df, path, first_write=True, compression='snappy'):
    table = pa.Table.from_pandas(df)
    if first_write:
        pq.write_table(table, path, compression=compression)
    else:
        existing = pq.read_table(path)
        combined = pa.concat_tables([existing, table])
        pq.write_table(combined, path, compression=compression)

# Construct list of data frames or something similar
def const_acoustic_features(src_files):
    row_features = []

    for triplet in src_files:
        lab_phone_feats = process_labeled_phonemes(triplet['ph'])

        # Retrieve samples from wav file
        samples, sr = librosa.load(triplet['wav'], sr=None)

        # Pull raw pitchmarks from file
        pms = []
        with open(triplet['pm'], "r", encoding="utf-8") as pm_file:
            pms = pm_file.readlines()
            pms = pms[8:]

        # For each phoneme do wav and pm processing
        for lab_phone_feat in lab_phone_feats:
            t_start = lab_phone_feat['start']
            t_dur   = lab_phone_feat['dur']

            # Wav processing
            c_samples = samples[int(t_start * sr) : int(t_start * sr + t_dur * sr)]
            lpow = log_power(c_samples)
            ceps = cepstrum(c_samples).reshape(-1)

            # Pitchmark processing
            lpitch = log_pitch(count_periods(pms, t_start, t_dur), t_dur)

            acoustic_features = {
                'log_power' : lpow,
                'log_pitch' : lpitch,
                'mfcc' : ceps.tolist(),
            }            
            row_features.append(lab_phone_feat | PHONEME_FEATURES[lab_phone_feat['ph']] | acoustic_features)        

    return pd.DataFrame(row_features)

# Batch calculate and write out features
def batch_features_from_files(data_src_dir, output_file,batchsize=100):
    src_files = gather_source_files(data_src_dir)
    for i in range(0, len(src_files) - batchsize, batchsize):
        feature_frames = const_acoustic_features(src_files[i : i + batchsize])
        write_parquet(feature_frames, output_file, first_write=(i == 0), compression='snappy')
    # Write out the training set features
    file_name = Path(output_file).stem + '_train.parquet'
    file_path = Path(output_file).parent
    feature_frames = const_acoustic_features(src_files[-batchsize:])
    write_parquet(feature_frames, file_path / file_name, first_write=True, compression='snappy')
    


In [38]:
# Run the feature production
# Notes: Looks like there's some outliers with pitch maximums.

batch_features_from_files('datasets/cmu_us_awb_arctic/', 'datasets/cmu_us_awb_arctic/feats/a0.parquet')